In [ ]:
# Config & Utils

import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

In [ ]:
load_dotenv()

PGUSER=os.getenv("PGUSER")
PGHOST=os.getenv("PGHOST")
PGDATABASE=os.getenv("PGDATABASE")
PGPASSWORD=os.getenv("PGPASSWORD")
PGPORT=os.getenv("PGPORT")


engine = create_engine(f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, engine)

def show(df, n=5):
    display(df.head(n))

def exec_sql(sql: str):
    """Execute DDL/DML SQL (no resultset)."""
    with engine.begin() as conn:
        conn.execute(text(sql))


In [ ]:
# --- CONFIG ---
SCHEMA = "public"
HORIZON_M = 12       # PD горизонт
ROLL_W = 3           # rolling window
SEQ_L = 12           # RNN history length
USE_SAMPLE = True    # включить подвыборку, чтобы не упираться в RAM
SAMPLE_N = 100000
SPLIT_MONTH = None

In [ ]:
# Проверка артефактов (views)

views = q(f"""
SELECT viewname
FROM pg_views
WHERE schemaname='{SCHEMA}'
ORDER BY viewname;
""")
show(views, 200)

required = {"fm_panel_base","fm_features_3m","fm_labels","fm_analysis","fm_baseline_12m"}
missing = required - set(views["viewname"])
missing

In [ ]:
# Санити-чек дат и строк

q(f"""
SELECT MIN(month) AS min_month, MAX(month) AS max_month, COUNT(*) AS n_rows
FROM {SCHEMA}.fm_panel_base
WHERE month IS NOT NULL;
""")

In [ ]:
# Основная статистика дефолтов (для статьи)

q(f"""
SELECT
  COUNT(*) AS n_loans,
  SUM(CASE WHEN censored=0 THEN 1 ELSE 0 END) AS n_defaulted,
  AVG(CASE WHEN censored=0 THEN 1.0 ELSE 0.0 END) AS default_rate
FROM {SCHEMA}.fm_labels;
""")

In [ ]:
# # Проверка событий (event) в survival-таблице
#
# q(f"""
# SELECT event, COUNT(*)
# FROM {SCHEMA}.fm_analysis
# GROUP BY event
# ORDER BY event;
# """)

In [ ]:
# # Проверка baseline PD(12m)
#
# q(f"""
# SELECT
#   COUNT(*) AS n_rows,
#   AVG(y12::float) AS y12_rate
# FROM {SCHEMA}.fm_baseline_12m;
# """)

In [ ]:
# График доли 30+/90+ по месяцам

tmp = q(f"""
SELECT
  month,
  AVG(CASE WHEN dq_m >= 1 THEN 1.0 ELSE 0.0 END) AS share_30plus,
  AVG(CASE WHEN dq_m >= 3 THEN 1.0 ELSE 0.0 END) AS share_90plus
FROM {SCHEMA}.fm_panel_base
WHERE month IS NOT NULL
GROUP BY month
ORDER BY month;
""")

plt.figure()
plt.plot(pd.to_datetime(tmp["month"]), tmp["share_30plus"])
plt.xticks(rotation=45)
plt.title("Доля наблюдений с dq_m ≥ 1 (≈30+ DPD) по месяцам")
plt.tight_layout()
plt.show()

plt.figure()
plt.plot(pd.to_datetime(tmp["month"]), tmp["share_90plus"])
plt.xticks(rotation=45)
plt.title("Доля наблюдений с dq_m ≥ 3 (≈90+ DPD) по месяцам")
plt.tight_layout()
plt.show()

In [ ]:
# “Треки” dq_m перед дефолтом (устойчиво к типам)

sample_def = q(f"""
SELECT loan_id, month, dq_m, default_month
FROM {SCHEMA}.fm_analysis
WHERE default_month IS NOT NULL;
""")

# обязательно приводим к datetime
sample_def["month"] = pd.to_datetime(sample_def["month"], errors="coerce")
sample_def["default_month"] = pd.to_datetime(sample_def["default_month"], errors="coerce")
sample_def = sample_def.dropna(subset=["month","default_month"])

sample_def["rel_m"] = ((sample_def["month"] - sample_def["default_month"]).dt.days // 30)
win = sample_def[(sample_def["rel_m"] >= -12) & (sample_def["rel_m"] <= 0)]
curve = win.groupby("rel_m")["dq_m"].mean().reset_index()

plt.figure()
plt.plot(curve["rel_m"], curve["dq_m"])
plt.title("Средний dq_m за 12 месяцев до дефолта (rel_m=0 — месяц dq_m≥3)")
plt.xlabel("Месяцы до дефолта")
plt.tight_layout()
plt.show()

In [ ]:
# Опционально: создаём/обновляем sample кредитов для лёгкой подготовки RNN-датасета
# Это решает проблему нехватки памяти (ячейка 8/10), не меняя логику исследования.

if USE_SAMPLE:
    # 1) sample таблица loan_id
    exists = q("SELECT to_regclass('public.fm_loan_sample') AS t").iloc[0,0]
    if exists is None:
        exec_sql(f"""
        CREATE TABLE public.fm_loan_sample AS
        SELECT loan_id
        FROM {SCHEMA}.fm_labels
        ORDER BY md5(loan_id)
        LIMIT {SAMPLE_N};
        """)
        exec_sql("CREATE INDEX IF NOT EXISTS ix_fm_loan_sample ON public.fm_loan_sample(loan_id)")
    else:
        # обновляем состав sample (воспроизводимо)
        exec_sql("TRUNCATE TABLE public.fm_loan_sample")
        exec_sql(f"""
        INSERT INTO public.fm_loan_sample(loan_id)
        SELECT loan_id
        FROM {SCHEMA}.fm_labels
        ORDER BY md5(loan_id)
        LIMIT {SAMPLE_N};
        """)

    # 2) материализованный источник для RNN (t и длина ряда считаются в SQL)
    mv_exists = q("SELECT to_regclass('public.fm_rnn_source_sample') AS t").iloc[0,0]
    if mv_exists is None:
        exec_sql(f"""
        CREATE MATERIALIZED VIEW public.fm_rnn_source_sample AS
        SELECT
          a.loan_id,
          a.month,
          a.event,
          a.dq_m, a.dq_change_1m, a.roll_worse_1m, a.cure_1m, a.dq_mean_3m, a.dq_std_3m,
          a.amort_buffer, a.deferral_amt, a.deferral_flag, a.mod_flag,
          a.rate, a.ltv, a.cltv, a.dti, a.fico_cur,
          a.sch_prin, a.tot_prin, a.unsch_prin, a.delin_int,
          (ROW_NUMBER() OVER (PARTITION BY a.loan_id ORDER BY a.month) - 1) AS t,
          COUNT(*) OVER (PARTITION BY a.loan_id) AS T_cnt
        FROM {SCHEMA}.fm_analysis a
        JOIN public.fm_loan_sample s USING (loan_id)
        WHERE a.month IS NOT NULL;
        """)
        exec_sql("CREATE INDEX IF NOT EXISTS ix_fm_rnn_source_sample_loan_month ON public.fm_rnn_source_sample(loan_id, month)")
    else:
        exec_sql("REFRESH MATERIALIZED VIEW public.fm_rnn_source_sample")

    print('Sample и MV готовы:', SAMPLE_N)
else:
    print('USE_SAMPLE=False: будет попытка загрузить весь fm_analysis (может не хватить RAM).')


In [ ]:
# Подготовка rnn_df (источник для RNN/hazard). Память экономим:
# 1) используем sample+materialized view,
# 2) t и длину ряда считаем в SQL, чтобы не делать тяжелый groupby в pandas,
# 3) даункастим типы.

features = [
    "dq_m","dq_change_1m","roll_worse_1m","cure_1m","dq_mean_3m","dq_std_3m",
    "amort_buffer","deferral_amt","deferral_flag","mod_flag",
    "rate","ltv","cltv","dti","fico_cur",
    "sch_prin","tot_prin","unsch_prin","delin_int"
]

if USE_SAMPLE:
    source = "public.fm_rnn_source_sample"
    sql = f"""
    SELECT loan_id, month, event, t, {",".join(features)}
    FROM {source}
    WHERE T_cnt >= {SEQ_L}
    ORDER BY loan_id, month;
    """
else:
    # Осторожно: полный объём может не поместиться в память.
    # Этот запрос корректен, но лучше использовать chunksize-экспорт (сделаем позже при необходимости).
    sql = f"""
    WITH z AS (
      SELECT
        loan_id, month, event,
        (ROW_NUMBER() OVER (PARTITION BY loan_id ORDER BY month) - 1) AS t,
        {",".join(features)},
        COUNT(*) OVER (PARTITION BY loan_id) AS T_cnt
      FROM {SCHEMA}.fm_analysis
      WHERE month IS NOT NULL
    )
    SELECT loan_id, month, event, t, {",".join(features)}
    FROM z
    WHERE T_cnt >= {SEQ_L}
    ORDER BY loan_id, month;
    """

rnn_df = q(sql)

# типы (month нужен для временного сплита; t — для последовательностей)
rnn_df["month"] = pd.to_datetime(rnn_df["month"], errors="coerce")
for c in features + ["event","t"]:
    rnn_df[c] = pd.to_numeric(rnn_df[c], errors="coerce")
rnn_df = rnn_df.dropna(subset=["month"])
rnn_df = rnn_df.fillna(0.0)

# даункаст типов — сильная экономия RAM
for c in features:
    rnn_df[c] = rnn_df[c].astype("float32")
rnn_df["event"] = rnn_df["event"].astype("int8")
rnn_df["t"] = rnn_df["t"].astype("int16")

# опционально: временной сплит для дальнейших расчетов (без утечки будущей информации)
if SPLIT_MONTH is not None:
    split_dt = pd.to_datetime(SPLIT_MONTH)
    train_df = rnn_df[rnn_df["month"] <= split_dt].copy()
    test_df  = rnn_df[rnn_df["month"] >  split_dt].copy()
    print("Train rows:", len(train_df), "Test rows:", len(test_df))
else:
    train_df, test_df = None, None

rnn_df.shape, int(rnn_df["event"].sum())

In [ ]:
# --- Sanity-check: event должен быть first-hit (0/1 на loan_id) и без строк после события ---

df_chk = rnn_df.copy()
df_chk["month"] = pd.to_datetime(df_chk["month"], errors="coerce")
df_chk = df_chk.dropna(subset=["month"]).sort_values(["loan_id", "month"])

# 1) Сколько event=1 на один loan_id
evt_cnt = df_chk.groupby("loan_id")["event"].sum().astype(int)
print("Распределение SUM(event) по loan_id (ожидаем только 0 и 1):")
print(evt_cnt.value_counts().sort_index().head(20))

bad_multi = evt_cnt[evt_cnt > 1]
print("\nloan_id с SUM(event) > 1:", len(bad_multi))
if len(bad_multi) > 0:
    print("Примеры (loan_id -> n_events):")
    print(bad_multi.sort_values(ascending=False).head(20))

# 2) Есть ли строки ПОСЛЕ первого события (утечка "после дефолта")
first_evt = (df_chk[df_chk["event"] == 1]
             .groupby("loan_id")["month"].min()
             .rename("first_evt_month"))
df2 = df_chk.merge(first_evt, on="loan_id", how="left")

after_evt = df2[(df2["first_evt_month"].notna()) & (df2["month"] > df2["first_evt_month"])]
print("\nСтрок после первого события (ожидаем 0):", len(after_evt))
if len(after_evt) > 0:
    print("Примеры loan_id с количеством строк после события:")
    print(after_evt.groupby("loan_id").size().sort_values(ascending=False).head(20))

# 3) Быстрый просмотр траектории для одного проблемного кредита (если есть)
if len(bad_multi) > 0:
    ex_id = bad_multi.sort_values(ascending=False).index[0]
    display(df2[df2["loan_id"] == ex_id][["loan_id","month","event","first_evt_month"]].head(50))
elif len(after_evt) > 0:
    ex_id = after_evt.groupby("loan_id").size().sort_values(ascending=False).index[0]
    display(df2[df2["loan_id"] == ex_id][["loan_id","month","event","first_evt_month"]].head(50))


In [ ]:
# Мини-итоги и сохранение подготовленного long-датасета для RNN (без обучения)
seq_stats = (rnn_df.groupby('loan_id')['t'].max() + 1).describe()
print('Длина последовательностей (месяцы) по loan_id:')
display(seq_stats)

# Сохраняем на диск, чтобы не пересчитывать и не держать всё время в RAM
# Parquet предпочтительнее, но если нет pyarrow/fastparquet — используйте pickle.
OUT_PATH = 'rnn_long_sample.parquet' if USE_SAMPLE else 'rnn_long_full.parquet'
try:
    rnn_df.to_parquet(OUT_PATH, index=False)
    print('Saved:', OUT_PATH)
except Exception as e:
    OUT_PATH = OUT_PATH.replace('.parquet','.pkl')
    rnn_df.to_pickle(OUT_PATH)
    print('Parquet not available, saved pickle:', OUT_PATH, '|', str(e)[:200])


In [ ]:
# # 6. Baseline и RNN (Neural Survival / Hazard)
#
# В этом блоке мы доводим пайплайн до логического конца:
# - формируем обучающую выборку в постановке **one-step discrete-time hazard** (прогноз дефолта в следующем месяце),
# - обучаем бенчмарки (Logit, GBDT),
# - обучаем **RNN (GRU/LSTM)** на последовательностях длиной `SEQ_L=12`,
# - рассчитываем метрики (ROC-AUC, PR-AUC, Brier), lift@k и строим графики для раздела *Results*.
#
# > Важно: дефолт определяется как **первое** вхождение в 90+ DPD (dq_m ≥ 3). Таргет строится на шаг вперёд: `y_next = first_default_{t+1}`. В обучение включаются только месяцы **до** первого дефолта (risk set).
#

# --- Imports for modeling ---
import random
import copy
from datetime import datetime

# sklearn metrics / baselines
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_curve, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

# torch (RNN)
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except Exception as e:
    raise ImportError("PyTorch не установлен. Установите torch (pip/conda) и перезапустите kernel.") from e

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


In [ ]:

# ## 6.1. Формирование таргета one-step hazard без «псевдо-дефолтов»
#
# Ключевой момент: **дефолт** в исследовании трактуется как *первое* вхождение в 90+ DPD (dq_m >= 3),
# а не «нахождение в дефолте» несколько месяцев подряд.
#
# Мы строим постановку **one-step discrete-time hazard**:
# для наблюдения в момент t целевая переменная равна 1, если *в следующий месяц* произойдёт первое вхождение в 90+ DPD.
#
# \[
#  y_{i,t}^{next} = \mathbb{1}\{\text{first 90+ DPD happens at } t+1\}.
# \]
#
# Дополнительно формируем *risk set*: в обучение/оценку включаются только месяцы **до** первого дефолта.
# Это убирает «тривиальные» наблюдения после дефолта и делает метрики интерпретируемыми как *early warning*.


# rnn_df ожидается из предыдущих ячеек (источник: fm_rnn_source_sample или full)
df_all = rnn_df.copy()

# сортировка — обязательна для корректных лагов/сдвигов
df_all = df_all.sort_values(["loan_id", "t"]).reset_index(drop=True)

# приведение типов (на всякий случай)
df_all["month"] = pd.to_datetime(df_all["month"], errors="coerce")
df_all["dq_m"] = pd.to_numeric(df_all["dq_m"], errors="coerce").fillna(0).astype(int)

# 1) event_first = 1 только в месяц ПЕРВОГО dq_m>=3
dq3 = df_all["dq_m"] >= 3
prev_dq3 = (df_all.groupby("loan_id")["dq_m"].shift(1).fillna(0).astype(int) >= 3)
df_all["event_first"] = (dq3 & (~prev_dq3)).astype("int8")

# таблица с месяцем первого дефолта (нужна для графиков Results)
default_month_tbl = (
    df_all.loc[df_all["event_first"] == 1, ["loan_id", "month"]]
    .groupby("loan_id", as_index=False)["month"].min()
    .rename(columns={"month": "default_month_first"})
)

# 2) t_first_default (в терминах индекса t) — для фильтрации risk set
t_first_tbl = (
    df_all.loc[df_all["event_first"] == 1, ["loan_id", "t"]]
    .groupby("loan_id", as_index=False)["t"].min()
    .rename(columns={"t": "t_first_default"})
)
df_all = df_all.merge(t_first_tbl, on="loan_id", how="left")
df_all["t_first_default"] = df_all["t_first_default"].fillna(np.inf)

# 3) y_next = event_first_{t+1}
df_all["y_next"] = df_all.groupby("loan_id")["event_first"].shift(-1)

# 4) risk set: берём только месяцы ДО первого дефолта
df_full = df_all[df_all["t"] < df_all["t_first_default"]].copy()

# исключаем последние месяцы (там нет t+1 -> y_next=NaN)
df_full = df_full.dropna(subset=["y_next"]).copy()
df_full["y_next"] = df_full["y_next"].astype("int8")

print("Rows in risk set with y_next:", len(df_full))
print("Unique loans:", df_full["loan_id"].nunique())
print("Positive rate (y_next):", float(df_full["y_next"].mean()))

# Подвыборка точек, где доступна история длиной SEQ_L (нужна для baselines и сопоставимой оценки с RNN)
df0 = df_full[df_full["t"] >= (SEQ_L - 1)].copy()
print("Rows with SEQ_L history (t>=L-1):", len(df0))


In [ ]:
# ## 6.2. Разбиение на train/val/test по loan_id (без утечки)
#
# Чтобы один и тот же кредит не попадал одновременно в обучение и тест, делим **по loan_id** (детерминированно через хеш).


import hashlib

def split_bucket(loan_id: str) -> int:
    h = hashlib.md5(str(loan_id).encode("utf-8")).hexdigest()
    return int(h[:8], 16) % 100

# назначаем сплит на уровне кредита
loan_ids = df_full["loan_id"].drop_duplicates()
buckets = loan_ids.map(split_bucket)
loan_split = pd.DataFrame({"loan_id": loan_ids.values, "bucket": buckets.values})
loan_split["split"] = np.where(loan_split["bucket"] < 80, "train",
                        np.where(loan_split["bucket"] < 90, "val", "test"))

df_full = df_full.merge(loan_split[["loan_id","split"]], on="loan_id", how="left")
df0      = df0.merge(loan_split[["loan_id","split"]], on="loan_id", how="left")

df0["split"].value_counts()

In [ ]:
# 6.3. Baseline модели (табличные): Snapshot vs History-aggregated (12m)

feature_cols = features  # из предыдущей части ноутбука

# на всякий случай пересоберём df0 из df_full (после split)
df0 = df_full[df_full["t"] >= (SEQ_L - 1)].copy()

# --- 1) Snapshot dataset ---
train = df0[df0["split"]=="train"].copy()
val   = df0[df0["split"]=="val"].copy()
test  = df0[df0["split"]=="test"].copy()

X_train = train[feature_cols].astype("float32").values
y_train = train["y_next"].values.astype(int)
X_val   = val[feature_cols].astype("float32").values
y_val   = val["y_next"].values.astype(int)
X_test  = test[feature_cols].astype("float32").values
y_test  = test["y_next"].values.astype(int)

def eval_binary(y_true, y_prob, name="model"):
    roc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true))>1 else np.nan
    pr  = average_precision_score(y_true, y_prob) if len(np.unique(y_true))>1 else np.nan
    br  = brier_score_loss(y_true, y_prob)
    return {"model": name, "roc_auc": roc, "pr_auc": pr, "brier": br}

# Logistic Regression (class-balanced)
logit = LogisticRegression(max_iter=300, class_weight="balanced", n_jobs=-1, solver="lbfgs")
logit.fit(X_train, y_train)
p_logit = logit.predict_proba(X_test)[:,1]

# HistGradientBoosting (встроенный GBDT без сторонних библиотек)
gbdt = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=200)
gbdt.fit(X_train, y_train)
p_gbdt = gbdt.predict_proba(X_test)[:,1]

# --- 2) History-aggregated dataset (12m mean/std) ---
# Важно: агрегаты считаем на df_full (включая всю историю до t), затем берём строки t>=L-1
df_roll = df_full.sort_values(["loan_id","t"]).copy()

for c in feature_cols:
    s = df_roll.groupby("loan_id")[c].rolling(SEQ_L, min_periods=SEQ_L)
    df_roll[f"{c}_mean12"] = s.mean().reset_index(level=0, drop=True).astype("float32")
    df_roll[f"{c}_std12"]  = s.std(ddof=0).reset_index(level=0, drop=True).fillna(0.0).astype("float32")

# выберем только точки, где агрегаты доступны
df0_hist = df_roll[df_roll["t"] >= (SEQ_L - 1)].copy()

hist_cols = []
for c in feature_cols:
    hist_cols += [f"{c}_mean12", f"{c}_std12"]
feature_cols_hist = feature_cols + hist_cols  # текущий срез + агрегаты

train_h = df0_hist[df0_hist["split"]=="train"].copy()
val_h   = df0_hist[df0_hist["split"]=="val"].copy()
test_h  = df0_hist[df0_hist["split"]=="test"].copy()

X_train_h = train_h[feature_cols_hist].astype("float32").fillna(0.0).values
y_train_h = train_h["y_next"].values.astype(int)
X_test_h  = test_h[feature_cols_hist].astype("float32").fillna(0.0).values
y_test_h  = test_h["y_next"].values.astype(int)

logit_h = LogisticRegression(max_iter=400, class_weight="balanced", n_jobs=-1, solver="lbfgs")
logit_h.fit(X_train_h, y_train_h)
p_logit_h = logit_h.predict_proba(X_test_h)[:,1]

gbdt_h = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=200)
gbdt_h.fit(X_train_h, y_train_h)
p_gbdt_h = gbdt_h.predict_proba(X_test_h)[:,1]

# метрики (baseline)
baseline_metrics = pd.DataFrame([
    eval_binary(y_test,   p_logit,   "Logit (snapshot)"),
    eval_binary(y_test,   p_gbdt,    "GBDT (snapshot)"),
    eval_binary(y_test_h, p_logit_h, "Logit (12m agg)"),
    eval_binary(y_test_h, p_gbdt_h,  "GBDT (12m agg)"),
])

# сохраним baseline-предсказания на test с метаданными (для честного merge с RNN)
baseline_pred_test = test[["loan_id","month","y_next"]].copy()
baseline_pred_test["month"] = pd.to_datetime(baseline_pred_test["month"], errors="coerce")
baseline_pred_test = baseline_pred_test.rename(columns={"y_next":"y_true"})
baseline_pred_test["p_logit_snap"] = p_logit
baseline_pred_test["p_gbdt_snap"]  = p_gbdt

# history-aggregated test может иметь тот же набор точек, но лучше мерджить по (loan_id, month)
tmp_h = test_h[["loan_id","month","y_next"]].copy()
tmp_h["month"] = pd.to_datetime(tmp_h["month"], errors="coerce")
tmp_h = tmp_h.rename(columns={"y_next":"y_true"})
tmp_h["p_logit_agg"] = p_logit_h
tmp_h["p_gbdt_agg"]  = p_gbdt_h

baseline_pred_test = baseline_pred_test.merge(
    tmp_h[["loan_id","month","p_logit_agg","p_gbdt_agg"]],
    on=["loan_id","month"],
    how="left"
)

baseline_metrics


In [ ]:
# 6.4. RNN-модель: GRU для one-step hazard

class WindowDataset(Dataset):
    """
    Dataset из скользящих окон длины seq_len.
    Доп. возможность: mask_idx — индексы признаков, которые нужно занулить (для group ablation).
    """
    def __init__(self, df_full, feature_cols, seq_len=12, split="train", mask_idx=None):
        self.feature_cols = feature_cols
        self.seq_len = seq_len
        self.mask_idx = mask_idx  # list[int] or None

        self.df = df_full[df_full["split"]==split].copy()
        self.df = self.df.sort_values(["loan_id","t"]).reset_index(drop=True)

        self.windows = []  # list of (group_index, end_pos_in_group)
        self.groups = []   # list of (loan_id, X, y, month)

        for loan_id, g in self.df.groupby("loan_id", sort=False):
            X = g[feature_cols].astype("float32").to_numpy()
            y = g["y_next"].astype("float32").to_numpy()
            month = pd.to_datetime(g["month"], errors="coerce").to_numpy()

            T = len(g)
            for end in range(self.seq_len-1, T):
                self.windows.append((len(self.groups), end))
            self.groups.append((loan_id, X, y, month))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        g_idx, end = self.windows[idx]
        loan_id, X, y, month = self.groups[g_idx]
        start = end - self.seq_len + 1

        # writable/contiguous для PyTorch
        x_seq = np.ascontiguousarray(X[start:end+1], dtype=np.float32)  # [L, F]

        # group ablation: зануляем выбранные признаки
        if self.mask_idx is not None and len(self.mask_idx) > 0:
            x_seq[:, self.mask_idx] = 0.0

        target = np.float32(y[end])  # scalar
        return torch.from_numpy(x_seq), torch.tensor(target, dtype=torch.float32)

    def get_meta(self, idx):
        g_idx, end = self.windows[idx]
        loan_id, X, y, month = self.groups[g_idx]
        return loan_id, month[end]


train_ds = WindowDataset(df_full, feature_cols, seq_len=SEQ_L, split="train")
val_ds   = WindowDataset(df_full, feature_cols, seq_len=SEQ_L, split="val")
test_ds  = WindowDataset(df_full, feature_cols, seq_len=SEQ_L, split="test")

len(train_ds), len(val_ds), len(test_ds)


class GRUHazard(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0 if num_layers==1 else dropout
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: [B, L, F]
        out, _ = self.gru(x)               # out: [B, L, H]
        last = out[:, -1, :]               # [B, H]
        logits = self.head(last).squeeze(-1)  # [B]
        return logits


def make_loader(ds, batch_size=256, shuffle=True):
    # важно: для predict/Results нужен shuffle=False
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, drop_last=False)


train_loader = make_loader(train_ds, batch_size=256, shuffle=True)
val_loader   = make_loader(val_ds, batch_size=512, shuffle=False)
test_loader  = make_loader(test_ds, batch_size=512, shuffle=False)

# pos_weight for imbalance
y_train_all = train["y_next"].values.astype(int)
pos = y_train_all.sum()
neg = len(y_train_all) - pos
pos_weight = torch.tensor([neg / max(pos, 1)], device=DEVICE, dtype=torch.float32)
print("pos:", int(pos), "neg:", int(neg), "pos_weight:", float(pos_weight))

model = GRUHazard(n_features=len(feature_cols), hidden_size=64).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def train_one_epoch(model, loader):
    model.train()
    total = 0.0
    n = 0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(y)
        n += len(y)
    return total / max(n, 1)


def predict_loader(model, loader, dataset=None, return_meta=False):
    """
    return_meta=True: вернёт loan_id/month через dataset.get_meta(idx)
    Работает корректно только если loader.shuffle=False
    """
    model.eval()
    all_y, all_p = [], []
    all_loan, all_month = [], []

    global_idx = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            logits = model(x)
            p = torch.sigmoid(logits).cpu().numpy()

            all_p.append(p)
            all_y.append(y.numpy())

            if return_meta:
                bs = len(y)
                for j in range(bs):
                    loan_id, m = dataset.get_meta(global_idx + j)
                    all_loan.append(loan_id)
                    all_month.append(m)
                global_idx += bs
            else:
                global_idx += len(y)

    y_true = np.concatenate(all_y).astype(int)
    y_prob = np.concatenate(all_p)
    if return_meta:
        return y_true, y_prob, all_loan, all_month
    return y_true, y_prob


best_val = -1
best_state = None

EPOCHS = 6
for epoch in range(1, EPOCHS+1):
    tr_loss = train_one_epoch(model, train_loader)
    yv, pv = predict_loader(model, val_loader)
    val_roc = roc_auc_score(yv, pv) if len(np.unique(yv))>1 else np.nan
    val_pr  = average_precision_score(yv, pv) if len(np.unique(yv))>1 else np.nan
    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_roc={val_roc:.4f} | val_pr={val_pr:.4f}")
    score = val_pr if not np.isnan(val_pr) else val_roc
    if score > best_val:
        best_val = score
        best_state = copy.deepcopy(model.state_dict())

if best_state is not None:
    model.load_state_dict(best_state)


In [ ]:
# 6.5. Оценка качества (test) и графики для Results

# --- RNN predictions on test (с метаданными) ---
yt, pt, loan_t, month_t = predict_loader(
    model, test_loader, dataset=test_ds, return_meta=True
)

pred_rnn = pd.DataFrame({
    "loan_id": loan_t,
    "month": pd.to_datetime(month_t, errors="coerce"),
    "y_true": yt,
    "p_rnn": pt
})

# --- Merge с baseline предсказаниями ---
merged = pred_rnn.merge(baseline_pred_test, on=["loan_id","month"], how="inner", suffixes=("_rnn","_base"))

# sanity-check: y_true должно совпадать
if "y_true_base" in merged.columns:
    mismatch = (merged["y_true_rnn"] != merged["y_true_base"]).mean()
    print("y mismatch share:", float(mismatch))
    merged["y_true"] = merged["y_true_rnn"].astype(int)
else:
    merged["y_true"] = merged["y_true"].astype(int)

print("Merged rows (common evaluation set):", len(merged))
print("Positive rate:", float(merged["y_true"].mean()))

# --- Metrics table ---
all_metrics = pd.DataFrame([
    eval_binary(merged["y_true"].values, merged["p_logit_snap"].values, "Logit (snapshot)"),
    eval_binary(merged["y_true"].values, merged["p_gbdt_snap"].values,  "GBDT (snapshot)"),
    eval_binary(merged["y_true"].values, merged["p_logit_agg"].values,  "Logit (12m agg)"),
    eval_binary(merged["y_true"].values, merged["p_gbdt_agg"].values,   "GBDT (12m agg)"),
    eval_binary(merged["y_true"].values, merged["p_rnn"].values,        "RNN-GRU (one-step hazard)"),
])
all_metrics


# --- Lift@k ---
def lift_at_k(y_true, y_prob, k=0.05):
    n = len(y_true)
    m = max(int(n*k), 1)
    order = np.argsort(-y_prob)
    top = order[:m]
    recall = y_true[top].sum() / max(y_true.sum(), 1)
    precision = y_true[top].mean()
    return recall, precision

rows=[]
for name, col in [
    ("Logit (snapshot)", "p_logit_snap"),
    ("GBDT (snapshot)",  "p_gbdt_snap"),
    ("Logit (12m agg)",  "p_logit_agg"),
    ("GBDT (12m agg)",   "p_gbdt_agg"),
    ("RNN-GRU",          "p_rnn"),
]:
    for k in [0.01, 0.05, 0.10]:
        rec, prec = lift_at_k(merged["y_true"].values, merged[col].values, k=k)
        rows.append({"model": name, "k": k, "recall@k": rec, "precision@k": prec})
lift_tbl = pd.DataFrame(rows)
lift_tbl


# --- Group ablation (financial logistics) ---
# Категоризация признаков по логистической концепции:
feature_groups = {
    "Nodes (borrower/contract state)": ["rate","ltv","cltv","dti","fico_cur"],
    "Arcs (payment flows)": ["sch_prin","tot_prin","unsch_prin","delin_int"],
    "Buffers (liquidity & relief)": ["amort_buffer","deferral_amt","deferral_flag"],
    "Disruptions/instability": ["dq_m","dq_change_1m","roll_worse_1m","cure_1m","dq_mean_3m","dq_std_3m","mod_flag"],
}
group_mask_idx = {
    g: [feature_cols.index(c) for c in cols if c in feature_cols]
    for g, cols in feature_groups.items()
}

base_pr = average_precision_score(yt, pt) if len(np.unique(yt))>1 else np.nan

abl_rows = []
for g, idx in group_mask_idx.items():
    # включаем маску
    test_ds.mask_idx = idx
    loader_mask = make_loader(test_ds, batch_size=512, shuffle=False)
    y_m, p_m = predict_loader(model, loader_mask)
    pr_m = average_precision_score(y_m, p_m) if len(np.unique(y_m))>1 else np.nan
    roc_m = roc_auc_score(y_m, p_m) if len(np.unique(y_m))>1 else np.nan
    abl_rows.append({"group": g, "mask_idx_n": len(idx), "pr_auc": pr_m, "roc_auc": roc_m, "delta_pr": pr_m - base_pr})

# reset
test_ds.mask_idx = None

ablation_rnn = pd.DataFrame(abl_rows).sort_values("delta_pr")
ablation_rnn


# --- (Optional) Baseline ablation on Logit (12m agg): retrain dropping group columns ---
def drop_cols_for_group(group_cols):
    cols = set(group_cols)
    cols |= {f"{c}_mean12" for c in group_cols}
    cols |= {f"{c}_std12" for c in group_cols}
    return cols

baseline_abl = []
full_pr = average_precision_score(y_test_h, p_logit_h) if len(np.unique(y_test_h))>1 else np.nan
baseline_abl.append({"group": "FULL", "pr_auc": full_pr})

for g, cols in feature_groups.items():
    drop = drop_cols_for_group(cols)
    keep = [c for c in feature_cols_hist if c not in drop]
    Xtr = train_h[keep].astype("float32").fillna(0.0).values
    ytr = train_h["y_next"].values.astype(int)
    Xte = test_h[keep].astype("float32").fillna(0.0).values
    yte = test_h["y_next"].values.astype(int)

    m = LogisticRegression(max_iter=400, class_weight="balanced", n_jobs=-1, solver="lbfgs")
    m.fit(Xtr, ytr)
    pte = m.predict_proba(Xte)[:,1]
    pr = average_precision_score(yte, pte) if len(np.unique(yte))>1 else np.nan
    baseline_abl.append({"group": f"drop: {g}", "pr_auc": pr, "delta_pr": pr - full_pr})

baseline_ablation_logit12 = pd.DataFrame(baseline_abl)
baseline_ablation_logit12


# --- Figures ---
import os
os.makedirs("outputs", exist_ok=True)

# PR curve (RNN)
prec, rec, _ = precision_recall_curve(merged["y_true"].values, merged["p_rnn"].values)
plt.figure()
plt.plot(rec, prec)
plt.title("PR curve: RNN-GRU (one-step hazard)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.tight_layout()
plt.savefig("outputs/fig_pr_rnn.png", dpi=200)
plt.show()

# ROC curve (RNN)
fpr, tpr, _ = roc_curve(merged["y_true"].values, merged["p_rnn"].values)
plt.figure()
plt.plot(fpr, tpr)
plt.title("ROC curve: RNN-GRU (one-step hazard)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.tight_layout()
plt.savefig("outputs/fig_roc_rnn.png", dpi=200)
plt.show()

# Lift chart (recall@k)
lift_plot = lift_tbl.pivot(index="k", columns="model", values="recall@k").reset_index()
plt.figure()
for col in [c for c in lift_plot.columns if c!="k"]:
    plt.plot(lift_plot["k"], lift_plot[col], label=col)
plt.title("Recall@k (top-k% highest risk)")
plt.xlabel("k")
plt.ylabel("Recall")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/fig_lift_recall.png", dpi=200)
plt.show()

# Group ablation (RNN): delta PR-AUC
plt.figure()
ablation_rnn_sorted = ablation_rnn.sort_values("delta_pr")
plt.barh(ablation_rnn_sorted["group"], ablation_rnn_sorted["delta_pr"])
plt.title("RNN group ablation: ΔPR-AUC after masking a feature group")
plt.xlabel("ΔPR-AUC (masked - base)")
plt.tight_layout()
plt.savefig("outputs/fig_rnn_group_ablation.png", dpi=200)
plt.show()


In [ ]:
# ## 6.5a. Диагностика «слишком идеального» lift
#
# Если recall@k=1 при малой доле дефолтов, precision@k становится почти фиксированным (prevalence/k).
# Ниже — быстрые проверки, чтобы отличить:
# (a) реальную почти-детерминированность (например, из-за сильных delinquency-фич),
# (b) возможную утечку/сдвиг меток (например, метка совпадает с текущим месяцем),
# (c) проблемы с выравниванием (merge) или оценкой.

import numpy as np
import pandas as pd

# 1) Базовая статистика
n = len(merged)
pos = int(merged['y_true'].sum())
prev = float(merged['y_true'].mean())
print(f"Eval N={n:,}, positives={pos:,}, prevalence={prev:.6%}")

# 2) Проверка: recall=1 => precision ~= prevalence/k
for k in [0.01, 0.05, 0.10]:
    m = max(int(n*k), 1)
    theo_prec = prev / k
    print(f"k={k:.2f}: top={m:,} | theoretical precision if recall=1: {theo_prec:.6f}")

# 3) Насколько модели похожи? (корреляция скорингов)
score_cols = ['p_logit_snap','p_gbdt_snap','p_logit_agg','p_gbdt_agg','p_rnn']
avail = [c for c in score_cols if c in merged.columns]
corr = merged[avail].corr(method='spearman')
print("\nSpearman correlation of score columns:")
print(corr.round(3))

# 4) Проверка на сдвиг меток: для y_true=1 ожидаем, что dq_m в текущем месяце ещё < 3,
# а dq_next (в следующем месяце) уже >=3. (если dq_m доступен)
if 'dq_m' in test.columns:
    tmp = test[['loan_id','month','dq_m']].copy()
    tmp = tmp.sort_values(['loan_id','month'])
    tmp['dq_next'] = tmp.groupby('loan_id')['dq_m'].shift(-1)
    merged_dbg = merged.merge(tmp, on=['loan_id','month'], how='left')
    print("\nDQ sanity-check (only if dq_m exists in test):")
    print("Share of positives where dq_m>=3 (should be ~0 if label is next-month default):",
          float((merged_dbg.loc[merged_dbg.y_true==1,'dq_m']>=3).mean()))
    print("Share of positives where dq_next>=3 (should be high):",
          float((merged_dbg.loc[merged_dbg.y_true==1,'dq_next']>=3).mean()))
    # распределение dq_m для positives
    print("\nDistribution dq_m for positives:")
    print(merged_dbg.loc[merged_dbg.y_true==1,'dq_m'].value_counts(dropna=False).sort_index())
else:
    merged_dbg = merged.copy()

# 5) Посмотрим top-1% по каждой модели и сколько там дефолтов
def topk_report(df, score_col, k=0.01):
    n=len(df)
    m=max(int(n*k),1)
    top = df.nlargest(m, score_col)
    rec = top['y_true'].sum()/max(df['y_true'].sum(),1)
    prec = top['y_true'].mean()
    return m, float(rec), float(prec)

print("\nTop-k reports:")
for col in avail:
    for k in [0.01,0.05,0.10]:
        m, rec, prec = topk_report(merged, col, k)
        print(f"{col:>12} | k={k:0.2f} top={m:>7,} recall={rec:0.3f} precision={prec:0.6f}")

# 6) Выгрузим несколько топ-наблюдений для ручной проверки
show_cols = ['loan_id','month','y_true'] + avail
if 'dq_m' in merged_dbg.columns:
    show_cols += ['dq_m','dq_next']

print("\nSample of top 15 rows by p_logit_snap:")
print(merged_dbg.sort_values('p_logit_snap', ascending=False)[show_cols].head(15))

# 7) Lift на «раннем предупреждении»: только точки, где текущая просрочка низкая (dq_m<=1)
if 'dq_m' in merged_dbg.columns:
    ew = merged_dbg[merged_dbg['dq_m'].fillna(0) <= 1].copy()
    print(f"\nEarly-warning subset (dq_m<=1): N={len(ew):,}, prevalence={ew['y_true'].mean():.6%}")
    for col in avail:
        for k in [0.01,0.05,0.10]:
            m, rec, prec = topk_report(ew, col, k)
            print(f"EW {col:>9} | k={k:0.2f} top={m:>7,} recall={rec:0.3f} precision={prec:0.6f}")


In [ ]:
# ## 6.5b. Diagnostics: why Lift@k looks "too good" and how to validate labels/calibration
#
# This cell addresses two common issues:
# (1) DQ sanity-check: dq_next computed on a filtered/merged evaluation set can be NaN.
#     We reconstruct dq_next from the original sequential panel (df_all) to validate that
#     y_next truly corresponds to "first 90+ DPD at t+1".
# (2) Probability saturation: class weighting (Logit) / pos_weight (RNN) improves ranking
#     but often breaks probability calibration. We quantify this and provide optional calibration.

import pandas as pd
import numpy as np

# --- 1) Reconstruct dq_next from the full sequential panel (df_all) ---
# df_all must contain one row per (loan_id, month) in temporal order and dq_m for that month.
dq_next_map = df_all[["loan_id","month","dq_m"]].copy()
dq_next_map["month"] = pd.to_datetime(dq_next_map["month"], errors="coerce")
dq_next_map = dq_next_map.rename(columns={"dq_m":"dq_next_true"})
# shift month backward so that dq at (t+1) aligns with row at month t
dq_next_map["month"] = dq_next_map["month"] - pd.DateOffset(months=1)

merged_dbg = merged.merge(dq_next_map, on=["loan_id","month"], how="left")

pos = merged_dbg[merged_dbg["y_true"]==1].copy()
print("Positives in eval set:", len(pos))
print("Share of positives with dq_next_true missing:", float(pos["dq_next_true"].isna().mean()))
print("Share of positives where dq_next_true>=3 (should be ~1.0):", float((pos["dq_next_true"]>=3).mean()))

print("\nDistribution (dq_m at t) among positives:")
if "dq_m" in merged_dbg.columns:
    print(pos["dq_m"].value_counts(dropna=False))
else:
    # if dq_m not merged into eval, compute it from df_all at month t
    dq_t_map = df_all[["loan_id","month","dq_m"]].copy()
    dq_t_map["month"] = pd.to_datetime(dq_t_map["month"], errors="coerce")
    merged_dbg = merged_dbg.merge(dq_t_map.rename(columns={"dq_m":"dq_m_t"}), on=["loan_id","month"], how="left")
    pos = merged_dbg[merged_dbg["y_true"]==1].copy()
    print(pos["dq_m_t"].value_counts(dropna=False))

# --- 2) How many positives by dq_m stage? (Explains why recall@1% can be 1.0) ---
# This helps interpret whether we are mainly predicting "60->90 DPD roll" (dq_m=2) vs earlier warning.
stage_counts = merged_dbg.groupby("dq_m", dropna=False)["y_true"].sum().sort_index() if "dq_m" in merged_dbg.columns else None
if stage_counts is not None:
    print("\nPositives by dq_m stage (t):")
    print(stage_counts)

# --- 3) Calibration warning for weighted models ---
# If Logit was trained with class_weight='balanced' (or RNN with pos_weight),
# predicted probabilities are typically NOT calibrated to the original event prevalence.
# Ranking metrics (ROC-AUC / PR-AUC / lift@k) remain informative, but PD interpretation requires calibration.
print("\nNOTE: If you use class weighting / pos_weight, treat probabilities as scores unless calibrated.")


In [ ]:
# подтянуть y_next из df_all на те же (loan_id, month)
y_next_map = df_all[["loan_id","month","y_next"]].copy()
y_next_map["month"] = pd.to_datetime(y_next_map["month"], errors="coerce")

merged_dbg2 = merged.merge(y_next_map, on=["loan_id","month"], how="left", suffixes=("","_from_full"))
print("Mismatch share y_true vs y_next_from_full:",
      (merged_dbg2["y_true"] != merged_dbg2["y_next"]).mean())


In [ ]:
# ## 6.5c. Cross-sectional lift (per calendar month) for PD12
#
# Чтобы метрики выглядели «как в бизнесе», можно считать lift не по всем loan-month сразу,
# а по каждому календарному месяцу отдельно: ранжируем активные кредиты в месяце t по PD12(t)
# и смотрим, какая доля будущих дефолтов (в пределах 12 мес) попадает в top-k%.

import numpy as np
import pandas as pd

if 'pd12_all' not in globals():
    print('pd12_all is not computed; run cell 6.5b first')
else:
    H=12
    df = pd12_all.copy()
    df = df.dropna(subset=[f'y{H}', f'pd{H}']).copy()
    df['month'] = pd.to_datetime(df['month'])

    def lift_monthly(g, k=0.01):
        y = g[f'y{H}'].values.astype(int)
        p = g[f'pd{H}'].values.astype(float)
        n=len(y)
        m=max(int(n*k),1)
        order=np.argsort(-p)
        top=order[:m]
        recall=y[top].sum()/max(y.sum(),1)
        precision=y[top].mean()
        return recall, precision, y.sum(), n

    rows=[]
    for model in df['model'].unique():
        dm = df[df['model']==model]
        for k in [0.01,0.05,0.10]:
            stats = dm.groupby('month').apply(lambda g: pd.Series(lift_monthly(g,k=k), index=['recall','precision','pos','n']))
            # усредняем recall, взвешивая по числу positives в месяце
            w = stats['pos'].values
            recall_w = np.average(stats['recall'].values, weights=np.where(w==0, 1, w))
            prec_w   = np.average(stats['precision'].values, weights=stats['n'].values)
            rows.append({'model': model, 'k': k, 'recall@k_monthly_PD12': float(recall_w), 'precision@k_monthly_PD12': float(prec_w)})

    lift_pd12_monthly = pd.DataFrame(rows)
    print('\n=== Monthly cross-sectional lift (PD12) ===')
    print(lift_pd12_monthly.to_markdown(index=False))
    lift_pd12_monthly.to_csv('outputs/lift_pd12_monthly_summary.csv', index=False)


In [ ]:
# 6.6. Динамика риска перед дефолтом: средний прогноз hazard по месяцам до события

pred_df = pred_rnn.copy()
pred_df["month"] = pd.to_datetime(pred_df["month"], errors="coerce")

# присоединим месяц первого дефолта, рассчитанный по event_first
default_month_tbl["default_month_first"] = pd.to_datetime(default_month_tbl["default_month_first"], errors="coerce")
pred_df = pred_df.merge(default_month_tbl, on="loan_id", how="left")

# оставим только дефолтные кредиты
pred_df = pred_df.dropna(subset=["default_month_first"]).copy()

# rel_m = month - default_month (в месяцах, отрицательные значения — до дефолта)
rel = (pred_df["month"].dt.to_period("M") - pred_df["default_month_first"].dt.to_period("M"))
pred_df["rel_m"] = rel.apply(lambda x: x.n).astype(int)

win = pred_df[(pred_df["rel_m"] >= -12) & (pred_df["rel_m"] <= -1)].copy()
curve = win.groupby("rel_m")["p_rnn"].mean().reset_index()

plt.figure()
plt.plot(curve["rel_m"], curve["p_rnn"])
plt.title("Средний one-step hazard за 12 месяцев до первого 90+ DPD (rel_m=-1 — месяц перед дефолтом)")
plt.xlabel("Месяцы до дефолта (rel_m)")
plt.ylabel("Средний прогноз hazard")
plt.tight_layout()
plt.savefig("outputs/fig_hazard_profile.png", dpi=200)
plt.show()

curve


# --- Also show PD(12m) profile (converted from hazard; locally-stationary approximation) ---
win["pd12_rnn"] = 1.0 - np.power(1.0 - win["p_rnn"].values, 12)
curve12 = win.groupby("rel_m")["pd12_rnn"].mean().reset_index()

plt.figure()
plt.plot(curve12["rel_m"], curve12["pd12_rnn"])
plt.title("Средний PD(12m) за 12 месяцев до первого 90+ DPD (конверсия из hazard)")
plt.xlabel("Месяцы до дефолта (rel_m)")
plt.ylabel("Средний прогноз PD(12m)")
plt.tight_layout()
plt.savefig("outputs/fig_pd12_profile.png", dpi=200)
plt.show()

curve12


Analytics

In [ ]:
# --- PD(12m) conversion (no leakage) ---
H = 12

def pd_horizon_from_hazard(p, H=12):
    p = np.clip(np.asarray(p, dtype="float64"), 0.0, 1.0)
    return 1.0 - np.power(1.0 - p, H)

# 1) Pull 12-month outcome label (y12) for the same (loan_id, month) points
y12_df = q(f'''
SELECT loan_id, month::date AS month, y12::int AS y12
FROM {SCHEMA}.fm_baseline_12m
''')
y12_df["month"] = pd.to_datetime(y12_df["month"], errors="coerce")

merged12 = merged.merge(y12_df, on=["loan_id","month"], how="left")
print("Merged for PD12. Rows:", len(merged12))
print("Share missing y12:", float(merged12["y12"].isna().mean()))
merged12 = merged12.dropna(subset=["y12"]).copy()
merged12["y12"] = merged12["y12"].astype(int)

# 2) Convert one-step probabilities -> PD12
prob_cols = {
    "Logit (snapshot)": "p_logit_snap",
    "GBDT (snapshot)":  "p_gbdt_snap",
    "Logit (12m agg)":  "p_logit_agg",
    "GBDT (12m agg)":   "p_gbdt_agg",
    "RNN-GRU (hazard)": "p_rnn",
}
for name, col in prob_cols.items():
    merged12[f"pd12_{col}"] = pd_horizon_from_hazard(merged12[col].values, H=H)

# 3) Metrics on PD12 horizon
pd12_metrics = pd.DataFrame([
    eval_binary(merged12["y12"].values, merged12["pd12_p_logit_snap"].values, "Logit (snapshot)"),
    eval_binary(merged12["y12"].values, merged12["pd12_p_gbdt_snap"].values,  "GBDT (snapshot)"),
    eval_binary(merged12["y12"].values, merged12["pd12_p_logit_agg"].values,  "Logit (12m agg)"),
    eval_binary(merged12["y12"].values, merged12["pd12_p_gbdt_agg"].values,   "GBDT (12m agg)"),
    eval_binary(merged12["y12"].values, merged12["pd12_p_rnn"].values,        "RNN-GRU (PD12 from hazard)"),
])
pd12_metrics

# 4) Lift@k for PD12
rows=[]
for name, col in [
    ("Logit (snapshot)", "pd12_p_logit_snap"),
    ("GBDT (snapshot)",  "pd12_p_gbdt_snap"),
    ("Logit (12m agg)",  "pd12_p_logit_agg"),
    ("GBDT (12m agg)",   "pd12_p_gbdt_agg"),
    ("RNN-GRU",          "pd12_p_rnn"),
]:
    for k in [0.01, 0.05, 0.10]:
        rec, prec = lift_at_k(merged12["y12"].values, merged12[col].values, k=k)
        rows.append({"model": name, "k": k, "recall@k": rec, "precision@k": prec})
lift_pd12 = pd.DataFrame(rows)
lift_pd12

# --- Save PD12 tables
import os
os.makedirs("outputs", exist_ok=True)
pd12_metrics.to_csv("outputs/metrics_pd12_summary.csv", index=False)
lift_pd12.to_csv("outputs/lift_pd12_summary.csv", index=False)

# --- Calibration plot (PD12): RNN vs best tabular baseline (by PR-AUC)
from sklearn.calibration import calibration_curve

# pick best baseline by PR-AUC
best_base_row = pd12_metrics[pd12_metrics["model"]!="RNN-GRU (PD12 from hazard)"].sort_values("pr_auc", ascending=False).head(1)
best_base_name = best_base_row["model"].iloc[0]
best_base_col = {
    "Logit (snapshot)":"pd12_p_logit_snap",
    "GBDT (snapshot)":"pd12_p_gbdt_snap",
    "Logit (12m agg)":"pd12_p_logit_agg",
    "GBDT (12m agg)":"pd12_p_gbdt_agg",
}[best_base_name]

plt.figure()
plt.plot([0,1],[0,1], linestyle="--", linewidth=1)

for label, pcol in [("RNN-GRU", "pd12_p_rnn"), (best_base_name, best_base_col)]:
    frac_pos, mean_pred = calibration_curve(merged12["y12"].values, merged12[pcol].values, n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac_pos, marker="o", label=label)

plt.title("Calibration: PD(12m) (quantile bins)")
plt.xlabel("Mean predicted PD12")
plt.ylabel("Observed default rate (12m)")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/fig_calibration_pd12.png", dpi=200)
plt.show()


In [ ]:
# --- RNN group ablation on PD(12m) horizon (interpretable importance) ---

from sklearn.metrics import average_precision_score, roc_auc_score

base_pr12 = average_precision_score(merged12["y12"].values, merged12["pd12_p_rnn"].values) if len(np.unique(merged12["y12"]))>1 else np.nan

abl12_rows = []
for g, idx in group_mask_idx.items():
    # mask group features
    test_ds.mask_idx = idx
    loader_mask = make_loader(test_ds, batch_size=512, shuffle=False)
    y_m, p_m, loan_m, month_m = predict_loader(model, loader_mask, dataset=test_ds, return_meta=True)

    pred_m = pd.DataFrame({
        "loan_id": loan_m,
        "month": pd.to_datetime(month_m, errors="coerce"),
        "p_mask": p_m
    })
    tmp = pred_m.merge(y12_df, on=["loan_id","month"], how="left").dropna(subset=["y12"]).copy()
    tmp["y12"] = tmp["y12"].astype(int)
    tmp["pd12_mask"] = pd_horizon_from_hazard(tmp["p_mask"].values, H=H)

    pr = average_precision_score(tmp["y12"].values, tmp["pd12_mask"].values) if len(np.unique(tmp["y12"]))>1 else np.nan
    roc = roc_auc_score(tmp["y12"].values, tmp["pd12_mask"].values) if len(np.unique(tmp["y12"]))>1 else np.nan
    abl12_rows.append({
        "group": g,
        "mask_idx_n": int(len(idx)),
        "pr_auc_pd12": float(pr) if pr==pr else np.nan,
        "roc_auc_pd12": float(roc) if roc==roc else np.nan,
        "delta_pr_pd12": (pr - base_pr12) if pr==pr else np.nan
    })

# reset
test_ds.mask_idx = None

ablation_rnn_pd12 = pd.DataFrame(abl12_rows).sort_values("delta_pr_pd12")
ablation_rnn_pd12.to_csv("outputs/rnn_group_ablation_pd12.csv", index=False)
ablation_rnn_pd12

# plot deltas (most negative first)
plt.figure()
plt.barh(ablation_rnn_pd12["group"], ablation_rnn_pd12["delta_pr_pd12"])
plt.title("RNN group ablation: impact on PR-AUC for PD(12m)")
plt.xlabel("Δ PR-AUC (masked - full)")
plt.tight_layout()
plt.savefig("outputs/fig_rnn_group_ablation_pd12.png", dpi=200)
plt.show()


In [ ]:
# --- Join selected explanatory variables for interpretable slicing ---
seg_cols = [
    "dq_m","dq_change_1m","roll_worse_1m","cure_1m","dq_mean_3m","dq_std_3m",
    "amort_buffer","deferral_flag","deferral_amt","mod_flag",
    "dti","ltv","fico_cur",
    "sch_prin","tot_prin","unsch_prin","delin_int"
]
seg_src = df0[["loan_id","month"] + [c for c in seg_cols if c in df0.columns]].copy()
seg_src["month"] = pd.to_datetime(seg_src["month"], errors="coerce")

ms = merged12.merge(seg_src, on=["loan_id","month"], how="left")

# --- Define interpretable states ---
def dq_bucket(v):
    if pd.isna(v):
        return "NA"
    v = int(v)
    if v <= 0:
        return "Current (0 DPD)"
    if v == 1:
        return "30+ DPD"
    if v == 2:
        return "60+ DPD"
    return "90+ DPD (pre-event)"

if "dq_m" in ms.columns:
    ms["dq_bucket"] = ms["dq_m"].apply(dq_bucket)
else:
    ms["dq_bucket"] = "NA"

if ("roll_worse_1m" in ms.columns) or ("dq_change_1m" in ms.columns):
    ms["shock"] = (
        (ms.get("roll_worse_1m", 0).fillna(0).astype(int) == 1) |
        (ms.get("dq_change_1m", 0).fillna(0) > 0)
    ).astype(int)
else:
    ms["shock"] = 0

# buffer quantiles (amort_buffer)
if "amort_buffer" in ms.columns:
    q1 = ms["amort_buffer"].quantile(0.25)
    q3 = ms["amort_buffer"].quantile(0.75)
    def buf_bucket(x):
        if pd.isna(x):
            return "NA"
        if x <= q1:
            return "Low buffer"
        if x >= q3:
            return "High buffer"
        return "Mid buffer"
    ms["buffer_bucket"] = ms["amort_buffer"].apply(buf_bucket)
else:
    ms["buffer_bucket"] = "NA"

# --- Summary table: PD12 by delinquency stage (disruptions) ---
grp1 = (ms.groupby("dq_bucket", dropna=False)
          .agg(n=("y12","size"),
               y12_rate=("y12","mean"),
               pd12_rnn=("pd12_p_rnn","mean"),
               pd12_best_base=(best_base_col,"mean"))
          .reset_index())
grp1.to_csv("outputs/pd12_by_dq_bucket.csv", index=False)
grp1

plt.figure()
x = np.arange(len(grp1))
plt.plot(x, grp1["y12_rate"].values, marker="o", label="Observed (12m)")
plt.plot(x, grp1["pd12_rnn"].values, marker="o", label="RNN-GRU")
plt.plot(x, grp1["pd12_best_base"].values, marker="o", label=f"Best baseline: {best_base_name}")
plt.xticks(x, grp1["dq_bucket"].values, rotation=20, ha="right")
plt.title("PD(12m) by disruption stage (delinquency state)")
plt.ylabel("Rate / PD12")
plt.tight_layout()
plt.legend()
plt.savefig("outputs/fig_pd12_by_dq_stage.png", dpi=200)
plt.show()

# --- Summary table: shock x buffer (actionable view) ---
grp2 = (ms.groupby(["shock","buffer_bucket"])
          .agg(n=("y12","size"),
               y12_rate=("y12","mean"),
               pd12_rnn=("pd12_p_rnn","mean"),
               pd12_best_base=(best_base_col,"mean"))
          .reset_index()
          .sort_values(["shock","buffer_bucket"]))
grp2.to_csv("outputs/pd12_shock_buffer_map.csv", index=False)
grp2

plt.figure()
labels = [f"S{int(r.shock)}-{r.buffer_bucket}" for r in grp2.itertuples()]
x = np.arange(len(labels))
plt.plot(x, grp2["y12_rate"].values, marker="o", label="Observed (12m)")
plt.plot(x, grp2["pd12_rnn"].values, marker="o", label="RNN-GRU")
plt.xticks(x, labels, rotation=45, ha="right")
plt.title("PD(12m) risk map: disruption (shock) vs buffer")
plt.ylabel("Rate / PD12")
plt.tight_layout()
plt.legend()
plt.savefig("outputs/fig_pd12_shock_buffer.png", dpi=200)
plt.show()


Outputs

In [ ]:

# ## 6.7. Экспорт результатов для статьи
#
# Сохраняем таблицы метрик, lift, а также ключевые графики для раздела *Results*.


all_metrics.to_csv("outputs/metrics_summary.csv", index=False)
pd12_metrics.to_csv("outputs/metrics_pd12_summary.csv", index=False)
lift_pd12.to_csv("outputs/lift_pd12_summary.csv", index=False)
lift_tbl.to_csv("outputs/lift_summary.csv", index=False)
ablation_rnn.to_csv("outputs/rnn_group_ablation.csv", index=False)

print("=== Metrics summary ===")
print(all_metrics.to_markdown(index=False))

print("\n=== Lift summary (recall@k, precision@k) ===")
print(lift_tbl.to_markdown(index=False))

print("\n=== RNN group ablation (masking) ===")
print(ablation_rnn.to_markdown(index=False))

print("\nФайлы сохранены в папку outputs/:")
print("- outputs/metrics_summary.csv")
print("- outputs/metrics_pd12_summary.csv")
print("- outputs/lift_pd12_summary.csv")
print("- outputs/pd12_by_dq_bucket.csv")
print("- outputs/pd12_shock_buffer_map.csv")
print("- outputs/fig_calibration_pd12.png")
print("- outputs/fig_pd12_by_dq_stage.png")
print("- outputs/fig_pd12_shock_buffer.png")
print("- outputs/lift_summary.csv")
print("- outputs/rnn_group_ablation.csv")
print("- outputs/fig_pr_rnn.png")
print("- outputs/fig_roc_rnn.png")
print("- outputs/fig_lift_recall.png")
print("- outputs/fig_rnn_group_ablation.png")
print("- outputs/rnn_group_ablation_pd12.csv")
print("- outputs/fig_rnn_group_ablation_pd12.png")
print("- outputs/fig_pd12_profile.png")
print("- outputs/fig_hazard_profile.png")
